##                                              IFT 511: ANALYSING BIG DATA
##                                                     PROJECT PART-2
##                                                 Prof. ASMA ELBADRAWY 

###                                                   Project Group-11
####                                               TANVI HEMANTBHAI PATEL
####                                              PRIYAM RAMESHBHAI MISTRI
####                                                    PRINCY PATEL

In [10]:
# We started by pulling in the basic tools we'll use throughout the notebook.
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix


In [11]:
# Here we load the raw CSV files. These are in the same folder as the notebook.
# The Book-Crossing data uses ';' instead of ',' as a separator.
ratings_df = pd.read_csv("Ratings.csv", sep=";")
books_df = pd.read_csv("Books.csv", sep=";")

# Clean up column names a bit (removed strange trailing spaces if any)
ratings_df.columns = ratings_df.columns.str.strip()
books_df.columns = books_df.columns.str.strip()

ratings_df.head(), books_df.head()


(   User-ID        ISBN  Rating
 0   276725  034545104X       0
 1   276726  0155061224       5
 2   276727  0446520802       0
 3   276729  052165615X       3
 4   276729  0521795028       6,
          ISBN                                              Title  \
 0  0195153448                                Classical Mythology   
 1  0002005018                                       Clara Callan   
 2  0060973129                               Decision in Normandy   
 3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
 4  0393045218                             The Mummies of Urumchi   
 
                  Author  Year                Publisher  
 0    Mark P. O. Morford  2002  Oxford University Press  
 1  Richard Bruce Wright  2001    HarperFlamingo Canada  
 2          Carlo D'Este  1991          HarperPerennial  
 3      Gina Bari Kolata  1999     Farrar Straus Giroux  
 4       E. J. W. Barber  1999   W. W. Norton & Company  )

In [12]:
# This dataset is huge, so we first narrow it down to users and books
# that have enough interactions to be meaningful.

# Keep users who have rated at least 65 books
user_counts = ratings_df["User-ID"].value_counts()
selected_users = user_counts[user_counts >= 65].index
filtered_ratings = ratings_df[ratings_df["User-ID"].isin(selected_users)]

# Keep books that received at least 65 ratings in this filtered set
book_counts = filtered_ratings["ISBN"].value_counts()
selected_books = book_counts[book_counts >= 65].index
filtered_ratings = filtered_ratings[filtered_ratings["ISBN"].isin(selected_books)]

# Turn the ratings into a user × book table:
# rows = users, columns = books, values = ratings (0 means "no rating")
user_book_matrix = filtered_ratings.pivot_table(
    index="User-ID",
    columns="ISBN",
    values="Rating",
    fill_value=0
)

print("User–Book matrix shape:", user_book_matrix.shape)
user_book_matrix.head()


User–Book matrix shape: (2567, 629)


ISBN,002542730X,0060008032,006016848X,0060391626,0060392452,0060502258,0060915544,0060921145,0060922532,0060928336,...,155874262X,1558743669,1558744150,1558744630,1558745157,1559029838,1573225789,1573229326,1573229725,1592400876
User-ID,,,,,,,,,,,,,,,,,,,,,
243,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
254,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
507,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
643,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# Each row in user_book_matrix is a vector of ratings for one user.
# We convert it to a sparse matrix then compute cosine similarity between all user pairs.

sparse_user_ratings = csr_matrix(user_book_matrix.values)

# similarity_matrix[u, v] will tell me how similar user u is to user v
similarity_matrix = cosine_similarity(sparse_user_ratings)

# Put the similarity values back into a DataFrame for easier lookup
user_ids = user_book_matrix.index
user_similarity = pd.DataFrame(similarity_matrix, index=user_ids, columns=user_ids)

user_similarity.head()


User-ID,243,254,507,638,643,741,882,1025,1211,1424,...,277195,277427,277478,277639,278026,278137,278144,278188,278418,278633
User-ID,,,,,,,,,,,,,,,,,,,,,
243,1.000000,0.000000,0.000000,0.108845,0.0,0.0,0.000000,0.0,0.000000,0.116985,...,0.000000,0.083818,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.077113
254,0.000000,1.000000,0.203382,0.000000,0.0,0.0,0.084467,0.0,0.000000,0.000000,...,0.157393,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
507,0.000000,0.203382,1.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
638,0.108845,0.000000,0.000000,1.000000,0.0,0.0,0.101987,0.0,0.307148,0.000000,...,0.000000,0.145296,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.110485
643,0.000000,0.000000,0.000000,0.000000,1.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [14]:
# Now we build the actual recommender:
# For each user, we look at their 10 closest neighbors and use their ratings
# (weighted by similarity) to predict scores for books the user has not read.

K_NEIGHBORS = 10
TOP_N_RECS = 5

recommendation_rows = []

for target_user in user_book_matrix.index:
    # 1) Similarity scores for this user to everyone else
    sim_scores = user_similarity[target_user].drop(target_user)  # drop self
    
    # 2) Pick the K_NEIGHBORS most similar users
    neighbor_ids = sim_scores.nlargest(K_NEIGHBORS).index.tolist()
    
    # 3) Books rated by at least one of these neighbors (this defines BK)
    neighbor_matrix = user_book_matrix.loc[neighbor_ids]
    
    # 4) Books the target user has not rated yet
    target_ratings = user_book_matrix.loc[target_user]
    not_rated_mask = (target_ratings == 0)
    candidate_book_ids = neighbor_matrix.columns[not_rated_mask]
    
    book_score_dict = {}
    
    # 5) For each candidate book, compute the weighted average rating
    for book_id in candidate_book_ids:
        weighted_sum = 0.0
        weight_total = 0.0
        
        for nb in neighbor_ids:
            nb_rating = user_book_matrix.at[nb, book_id]
            if nb_rating > 0:
                sim = user_similarity.at[target_user, nb]
                weighted_sum += nb_rating * sim
                weight_total += sim
        
        if weight_total > 0:
            estimated_score = weighted_sum / weight_total
            book_score_dict[book_id] = estimated_score
    
    # 6) Choose the TOP_N_RECS books with the highest estimated scores
    if book_score_dict:
        sorted_books = sorted(book_score_dict.items(), key=lambda x: x[1], reverse=True)
        top_books = sorted_books[:TOP_N_RECS]
        
        # 7) Attach book titles and store the results
        for book_id, score in top_books:
            # Try to fetch a readable title; if missing, fall back to 'Unknown'
            title_match = books_df.loc[books_df["ISBN"] == book_id, "Title"]
            if len(title_match) > 0:
                title_text = title_match.values[0]
            else:
                title_text = "Unknown"
            
            recommendation_rows.append([
                target_user,
                book_id,
                title_text,
                score
            ])

print("Finished computing recommendations for all users.")


Finished computing recommendations for all users.


In [15]:
# We now collect everything into a single DataFrame and write it out
# in the format requested by the assignment.

recommendations_table = pd.DataFrame(
    recommendation_rows,
    columns=["User_ID", "Book_ID", "Book_Title", "Recommendation_Score"]
)

recommendations_table.to_csv("Books_Recommended.csv", index=False)

print("Saved recommendations to 'Books_Recommended.csv'.")
recommendations_table.head()

Saved recommendations to 'Books_Recommended.csv'.


,User_ID,Book_ID,Book_Title,Recommendation_Score
0,243,0060921145,Animal Dreams,10.0
1,243,0060928336,Divine Secrets of the Ya-Ya Sisterhood: A Novel,10.0
2,243,080410753X,The Kitchen God's Wife,10.0
3,243,140003065X,A Fine Balance,10.0
4,243,0060959037,Prodigal Summer: A Novel,9.0


### Project Overview
In this part of our project, our group built a simple book recommendation system using collaborative filtering. The idea behind our approach is that people who tend to rate books similarly are likely to enjoy similar titles. By analyzing how users rate different books, we can estimate which books each person might appreciate next.
The outcome of our work is a file named Books_Recommended.csv, which contains the five recommended books for each user based on the method described below.

### Preparing the Data
The Book-Crossing dataset is quite large, so we first trimmed it to make the analysis manageable and more meaningful. We narrowed the data down to:
- Users who have rated at least 65 books, and
- Books that have received at least 65 ratings.

This filtering helps ensure that:
- We are working with users who have enough rating history to compare meaningfully.
- We focus on books that have enough feedback from readers.

After filtering, we converted the remaining ratings into a table where:
- rows represent individual users,
- columns represent books (ISBNs),
- and the values are the ratings.
Any missing ratings were replaced with 0.

### Understanding User Similarity
To compare users, we represented each one as a vector of their book ratings.
Then we calculated how similar each pair of users is based on the shape of their rating patterns.

This generated a full table showing how closely each user resembles every other user.
Having these similarity values allowed us to identify the users whose reading choices align most closely.

### Choosing Neighboring Users (K = 10)
For every user in the dataset, we took the following steps:
- Looked up how similar that user is to everyone else.
- Ranked the similarities.
- Selected the top 10 users who appear to have the closest reading behavior.

These 10 users form a kind of “peer group” for the target user.

### Estimating Scores for Unread Books
Once the set of nearest neighbors was identified, we gathered all the books these neighbors had rated.
This set of books forms the pool of possible recommendations.

For each book the target user had not rated, we estimated a score using a weighted approach:
- Ratings from more similar users carry more influence.
- Ratings from less similar users contribute less.

This gives us a predicted score that reflects how much the target user might like each book, based on the opinions of similar readers.

### Selecting Recommendations
After calculating estimated scores, we filtered out any books the user had already rated.
From the remaining list, we selected the five books with the highest predicted scores.

For each recommended book, we also matched the corresponding title from Books.csv, so the output would be easy to interpret.

### Final Output
All the recommendations were collected into a table with the following columns:
- User_ID
- Book_ID
- Book_Title
- Recommendation_Score

This table was finally saved as Books_Recommended.csv, which is the required output for this task.